# 01 · Quickstart & layer mapping

Register a model once, score it through a tiny benchmark, and record from the brain three ways: a single region (standard), the whole brain (`'all'`), and a population gathered across layers (`CompositeSelector`).

This notebook is laptop-safe: it uses a deterministic in-memory vision stand-in. The production CLIP × ROAR/MajajHong version is EC2-only, but the contract path shown here is the same `BrainScoreModel` layer-recording machinery.


## Load a small VLM and score one benchmark

In [1]:
import warnings
warnings.filterwarnings('ignore', message='xarray subclass Score should explicitly define __slots__', category=FutureWarning)
warnings.filterwarnings('ignore', message='unique with argument.*', category=FutureWarning)

import numpy as np
import pandas as pd
from brainscore_core.metrics import Score
from brainscore_core.model_interface import BrainScoreModel
from brainscore_core.supported_data_standards.brainio.assemblies import NeuroidAssembly
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

class ToyVisionExtractor:
    identifier = 'toy-vision-extractor'

    def __call__(self, stimuli, layers=None, **kwargs):
        layers = list(layers or ['layer4'])
        n_presentations = len(stimuli)
        blocks, layer_coord, neuroid_ids = [], [], []
        row_signal = np.arange(n_presentations, dtype=float)[:, None]
        for layer_i, layer in enumerate(layers):
            units = row_signal + np.array([[0.1, 0.2, 0.3]]) + layer_i
            blocks.append(units)
            layer_coord.extend([layer] * units.shape[1])
            neuroid_ids.extend([f'{layer}.u{j}' for j in range(units.shape[1])])
        data = np.concatenate(blocks, axis=1)
        return NeuroidAssembly(
            data,
            dims=['presentation', 'neuroid'],
            coords={
                'stimulus_id': ('presentation', list(stimuli['stimulus_id'])),
                'image_label': ('presentation', list(stimuli['image_label'])),
                'neuroid_id': ('neuroid', neuroid_ids),
                'layer': ('neuroid', layer_coord),
            },
        )

stimuli = StimulusSet(pd.DataFrame({
    'stimulus_id': ['s0', 's1', 's2', 's3'],
    'image_file_name': ['toy0.png', 'toy1.png', 'toy2.png', 'toy3.png'],
    'image_label': ['cat', 'dog', 'cat', 'dog'],
}))
stimuli.identifier = 'toy-images'

model = BrainScoreModel(
    identifier='toy-vision-model',
    model=None,
    region_layer_map={'V1': 'stem', 'V4': 'block3', 'IT': 'layer4'},
    preprocessors={'vision': ToyVisionExtractor()},
)

class ToyBenchmark:
    identifier = 'toy.IT.mean'
    required_modalities = {'vision'}

    def __call__(self, subject):
        subject.start_recording('IT')
        assembly = subject.process(stimuli)
        return Score(float(assembly.mean()))

benchmark = ToyBenchmark()
score = benchmark(model)
print('modalities:', model.supported_modalities)
print('toy benchmark score:', round(float(score), 3))


modalities: {'vision'}
toy benchmark score: 1.7


## Standard recording — one region maps to one layer

In [2]:
from brainscore_core.supported_data_standards.brainio.assemblies import walk_coords

def coord_values(assembly, name):
    for coord_name, _dims, values in walk_coords(assembly):
        if coord_name == name:
            return list(values)
    return []

print('region_layer_map:', dict(model.region_layer_map))
model.start_recording('IT')
assembly = model.process(stimuli)
print('recording region:', model._recording_regions,
      '-> layer(s):', model._recording_layers)
print('assembly shape:', dict(assembly.sizes))
print('assembly layers:', sorted(set(coord_values(assembly, 'layer'))))


region_layer_map: {'V1': 'stem', 'V4': 'block3', 'IT': 'layer4'}
recording region: ['IT'] -> layer(s): ['layer4']
assembly shape: {'presentation': 4, 'neuroid': 3}
assembly layers: ['layer4']


## Whole-brain recording — `start_recording('all')`
Every region in the model's `region_layer_map` is recorded in one pass; neuroids carry a `region` coord.

In [3]:
model.start_recording('all')
whole = model.process(stimuli)
print('regions:', model._recording_regions)
print('layers (deduped):', model._recording_layers)
print('whole-brain shape:', dict(whole.sizes))
print('whole-brain layers:', sorted(set(coord_values(whole, 'layer'))))


regions: ['V1', 'V4', 'IT']
layers (deduped): ['stem', 'block3', 'layer4']
whole-brain shape: {'presentation': 4, 'neuroid': 9}
whole-brain layers: ['block3', 'layer4', 'stem']


## Composite recording — a population across layers
`CompositeSelector` gathers units from several layers into one region — the v1.5 mechanism behind functional populations that span depth.

In [4]:
from brainscore_core import CompositeSelector

picked = ['stem', 'layer4']
sel = CompositeSelector(layers=tuple((layer, None) for layer in picked))
model._region_layer_selectors['Vc'] = sel
model._region_layer_map_dict['Vc'] = '|'.join(picked)
model.start_recording('Vc')
composite = model.process(stimuli)
print('composite layer paths:', sel.layer_paths)
print('composite recording:', model._composite_recording,
      '| layers:', model._recording_layers)
print('composite shape:', dict(composite.sizes))
print('composite layers:', sorted(set(coord_values(composite, 'layer'))))


composite layer paths: ('stem', 'layer4')
composite recording: True | layers: ['stem', 'layer4']
composite shape: {'presentation': 4, 'neuroid': 6}
composite layers: ['layer4', 'stem']


## Reset state (every notebook leaves the model clean)

In [5]:
model.reset()
print('recording state after reset:', model._recording_regions, model._recording_layers)


recording state after reset: [] []
